In [4]:
import torch
import torch.nn as nn

cnn = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
    nn.ReLU(),
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
)
print(cnn)

x = torch.rand(6, 3, 224, 224)

print(f"Input: {tuple(x.shape)}")
out = x
for i, layer in enumerate(cnn):
    out = layer(out)
    print(f"Layer {i} ({layer.__class__.__name__}): {tuple(out.shape)}")

print(f"\nFinal output shape: {tuple(out.shape)}")

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (1): ReLU()
  (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
Input: (6, 3, 224, 224)
Layer 0 (Conv2d): (6, 32, 112, 112)
Layer 1 (ReLU): (6, 32, 112, 112)
Layer 2 (Conv2d): (6, 64, 112, 112)
Layer 3 (BatchNorm2d): (6, 64, 112, 112)
Layer 4 (ReLU): (6, 64, 112, 112)
Layer 5 (MaxPool2d): (6, 64, 56, 56)

Final output shape: (6, 64, 56, 56)


In [5]:
def make_hook(name):
    def hook(module, input, output):
        print(f"{name} ({module.__class__.__name__}): {tuple(output.shape)}")
    return hook

for i, layer in enumerate(cnn):
    layer.register_forward_hook(make_hook(f"Layer {i}"))

x = torch.rand(1, 3, 224, 224)
out = cnn(x)   # hooks fire automatically during this call
print(f"\nFinal output shape: {tuple(out.shape)}")

Layer 0 (Conv2d): (1, 32, 112, 112)
Layer 1 (ReLU): (1, 32, 112, 112)
Layer 2 (Conv2d): (1, 64, 112, 112)
Layer 3 (BatchNorm2d): (1, 64, 112, 112)
Layer 4 (ReLU): (1, 64, 112, 112)
Layer 5 (MaxPool2d): (1, 64, 56, 56)

Final output shape: (1, 64, 56, 56)


In [6]:
# pip install torchinfo
from torchinfo import summary
summary(cnn, input_size=(1, 3, 224, 224))

Layer 0 (Conv2d): (1, 32, 112, 112)
Layer 1 (ReLU): (1, 32, 112, 112)
Layer 2 (Conv2d): (1, 64, 112, 112)
Layer 3 (BatchNorm2d): (1, 64, 112, 112)
Layer 4 (ReLU): (1, 64, 112, 112)
Layer 5 (MaxPool2d): (1, 64, 56, 56)


Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [1, 64, 56, 56]           --
├─Conv2d: 1-1                            [1, 32, 112, 112]         896
├─ReLU: 1-2                              [1, 32, 112, 112]         --
├─Conv2d: 1-3                            [1, 64, 112, 112]         18,496
├─BatchNorm2d: 1-4                       [1, 64, 112, 112]         128
├─ReLU: 1-5                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-6                         [1, 64, 56, 56]           --
Total params: 19,520
Trainable params: 19,520
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 243.25
Input size (MB): 0.60
Forward/backward pass size (MB): 16.06
Params size (MB): 0.08
Estimated Total Size (MB): 16.74